# 13 — 非芳香族環コンフォーマー比較 / Non-aromatic Ring Conformer Comparison

RDKit の **multi-seed EmbedMultipleConfs** と
Gypsum-DL 流の **random-coords + k-means クラスタリング** を比較する。

| 手法 | アルゴリズム |
|---|---|
| `rdkit_multi` | ETKDGv3 × N seeds → MMFF94 最適化 |
| `gypsum_style` | ETKDGv3 random start × thoroughness×N → MMFF94 → k-means on ring atoms |

比較指標：
- **エネルギー分布** (MMFF94 kcal/mol)
- **RMSD 多様性** (全ペア RMSD の最大値・平均値)
- **椅子/舟形の検出率** — シクロヘキサン環の Cremer-Pople 振幅 Q で判定


In [ ]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import AllChem

# NOTE: docking runners and preparation tools (gridbox, ligand, receptor)
# will be available in mdatools.docking in a future release.
from docking_analysis.preparation.ligand import (
    generate_conformers_multi,
    generate_ring_conformers_gypsum_style,
    compute_pairwise_conformer_rmsd,
    _get_nonaromatic_ring_atom_indices,
)

warnings.filterwarnings('ignore')


## テスト分子 / Test molecules

In [ ]:
TEST_MOLS = {
    "cyclohexane":    "C1CCCCC1",
    "piperidine":     "C1CCNCC1",
    "cyclohexanone":  "O=C1CCCCC1",
    "morpholine":     "C1COCCN1",
    "decalin":        "C1CCC2CCCCC2C1",
    "adamantane":     "C1C2CC3CC1CC(C2)C3",
    "methylcyclohex": "CC1CCCCC1",
    "piperazine":     "C1CNCCN1",
}

N_VARIANTS   = 6   # コンフォーマー数
THOROUGHNESS = 4   # Gypsum-DL スタイルの候補数倍率


## コンフォーマー生成 / Generate conformers

In [ ]:
def mmff_energy(mol_3d):
    props = AllChem.MMFFGetMoleculeProperties(mol_3d)
    if props is None:
        return None
    ff = AllChem.MMFFGetMoleculeForceField(mol_3d, props)
    return ff.CalcEnergy() if ff is not None else None


def max_pairwise_rmsd(confs):
    if len(confs) < 2:
        return 0.0
    mol_m = Chem.RWMol(confs[0])
    for c in confs[1:]:
        mol_m.AddConformer(c.GetConformer(0), assignId=True)
    mat = compute_pairwise_conformer_rmsd(mol_m.GetMol())
    return float(mat.max())


def mean_pairwise_rmsd(confs):
    if len(confs) < 2:
        return 0.0
    mol_m = Chem.RWMol(confs[0])
    for c in confs[1:]:
        mol_m.AddConformer(c.GetConformer(0), assignId=True)
    mat = compute_pairwise_conformer_rmsd(mol_m.GetMol())
    n = mat.shape[0]
    return float(mat[np.triu_indices(n, k=1)].mean())


results = []
confs_store = {}

for name, smi in TEST_MOLS.items():
    mol = Chem.MolFromSmiles(smi)
    confs_rdkit  = generate_conformers_multi(mol, n_seeds=N_VARIANTS)
    confs_gypsum = generate_ring_conformers_gypsum_style(
        mol, max_variants=N_VARIANTS, thoroughness=THOROUGHNESS
    )
    confs_store[name] = {'rdkit_multi': confs_rdkit, 'gypsum_style': confs_gypsum}

    for method, confs in [('rdkit_multi', confs_rdkit), ('gypsum_style', confs_gypsum)]:
        energies = [mmff_energy(c) for c in confs]
        energies = [e for e in energies if e is not None]
        results.append({
            'molecule':  name,
            'method':    method,
            'n_confs':   len(confs),
            'e_min':     min(energies) if energies else None,
            'e_max':     max(energies) if energies else None,
            'e_range':   max(energies) - min(energies) if len(energies) > 1 else 0.0,
            'max_rmsd':  max_pairwise_rmsd(confs),
            'mean_rmsd': mean_pairwise_rmsd(confs),
        })

df = pd.DataFrame(results)
print(df.to_string(index=False))


## エネルギー範囲・RMSD 多様性の比較 / Energy range and RMSD diversity

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

molecules = list(TEST_MOLS.keys())
x = np.arange(len(molecules))
width = 0.35

for ax, metric, ylabel in zip(
    axes,
    ['e_range', 'max_rmsd'],
    ['Energy range (kcal/mol)', 'Max pairwise RMSD (Å)'],
):
    rdkit_vals  = df[df.method == 'rdkit_multi' ][metric].values
    gypsum_vals = df[df.method == 'gypsum_style'][metric].values

    bars_r = ax.bar(x - width/2, rdkit_vals,  width, label='rdkit_multi',  color='steelblue',  alpha=0.8)
    bars_g = ax.bar(x + width/2, gypsum_vals, width, label='gypsum_style', color='darkorange', alpha=0.8)

    ax.set_xticks(x)
    ax.set_xticklabels(molecules, rotation=30, ha='right')
    ax.set_ylabel(ylabel)
    ax.set_title(ylabel)
    ax.legend()
    ax.bar_label(bars_r,  fmt='%.2f', fontsize=7)
    ax.bar_label(bars_g,  fmt='%.2f', fontsize=7)

plt.suptitle(
    f'Ring conformer comparison: rdkit_multi vs gypsum_style  '
    f'(N={N_VARIANTS}, thoroughness={THOROUGHNESS})',
    fontsize=11,
)
plt.tight_layout()
plt.savefig('ring_conformer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: ring_conformer_comparison.png')


## シクロヘキサン: Cremer-Pople 振幅 Q による椅子/舟形検出

Cremer-Pople の Q（全体振れ幅）が大きければ椅子形、小さければ平面/半椅子/舟形の指標。
- Chair: Q ≈ 0.55 Å
- Twist-boat / half-chair: Q ≈ 0.3–0.5 Å
- Flat / boat: Q < 0.3 Å


In [ ]:
def cremer_pople_q(mol_3d, ring_atom_indices):
    pos = mol_3d.GetConformer(0).GetPositions()
    coords = pos[ring_atom_indices]
    center = coords.mean(axis=0)
    coords_c = coords - center
    _, _, vh = np.linalg.svd(coords_c)
    normal = vh[-1]
    z = coords_c @ normal
    return float(np.sqrt(np.sum(z ** 2)))


def classify_ring(q):
    if q > 0.50:
        return 'chair'
    if q > 0.30:
        return 'twist/half-chair'
    return 'flat/boat'


rows = []
for method in ['rdkit_multi', 'gypsum_style']:
    for i, c in enumerate(confs_store['cyclohexane'][method]):
        ring_idx = _get_nonaromatic_ring_atom_indices(c)
        q = cremer_pople_q(c, ring_idx)
        e = mmff_energy(c)
        rows.append({
            'method':             method,
            'conf_idx':           i,
            'Q (Å)':              round(q, 3),
            'classification':     classify_ring(q),
            'energy (kcal/mol)':  round(e, 2) if e is not None else None,
        })

df_cx = pd.DataFrame(rows)
print(df_cx.to_string(index=False))


## シクロヘキサン: 椅子/非椅子の割合

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for ax, method in zip(axes, ['rdkit_multi', 'gypsum_style']):
    sub = df_cx[df_cx.method == method]
    counts = sub['classification'].value_counts()
    ax.pie(counts.values, labels=counts.index, autopct='%1.0f%%',
           colors=['#4C72B0', '#DD8452', '#55A868'])
    ax.set_title(f'{method}\n(cyclohexane, N={N_VARIANTS})')

plt.suptitle('Ring conformation classification by Cremer-Pople Q', fontsize=12)
plt.tight_layout()
plt.show()


## RMSD 行列ヒートマップ (デカリン) / RMSD heatmap (decalin)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

for ax, method in zip(axes, ['rdkit_multi', 'gypsum_style']):
    confs = confs_store['decalin'][method]
    mol_m = Chem.RWMol(confs[0])
    for c in confs[1:]:
        mol_m.AddConformer(c.GetConformer(0), assignId=True)
    mat = compute_pairwise_conformer_rmsd(mol_m.GetMol())

    im = ax.imshow(mat, cmap='YlOrRd', vmin=0)
    plt.colorbar(im, ax=ax, label='RMSD (Å)')
    ax.set_title(f'decalin — {method}')
    ax.set_xlabel('conformer index')
    ax.set_ylabel('conformer index')
    n = mat.shape[0]
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))

plt.suptitle('Pairwise RMSD matrix', fontsize=12)
plt.tight_layout()
plt.show()


## エネルギー vs RMSD from lowest-energy conformer

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for ax, mol_name in zip(axes.flatten(), TEST_MOLS.keys()):
    for method, color, marker in [
        ('rdkit_multi',  'steelblue',  'o'),
        ('gypsum_style', 'darkorange', 's'),
    ]:
        confs = confs_store[mol_name][method]
        energies = np.array([mmff_energy(c) for c in confs], dtype=float)
        valid = ~np.isnan(energies)
        energies = energies[valid]
        confs_v = [c for c, v in zip(confs, valid) if v]
        if len(confs_v) < 2:
            continue
        e_min_idx = int(np.argmin(energies))
        ref = confs_v[e_min_idx]

        rmsds = []
        for c in confs_v:
            mol_pair = Chem.RWMol(ref)
            mol_pair.AddConformer(c.GetConformer(0), assignId=True)
            mat = compute_pairwise_conformer_rmsd(mol_pair.GetMol())
            rmsds.append(mat[0, 1])

        ax.scatter(rmsds, energies - energies.min(),
                   label=method, color=color, marker=marker, alpha=0.8, s=60)

    ax.set_title(mol_name)
    ax.set_xlabel('RMSD from lowest-E (Å)')
    ax.set_ylabel('ΔE (kcal/mol)')
    ax.legend(fontsize=7)

plt.suptitle('Energy vs RMSD from lowest-energy conformer', fontsize=12)
plt.tight_layout()
plt.show()


## サマリーテーブル / Summary table

In [ ]:
pivot = df.pivot_table(
    index='molecule',
    columns='method',
    values=['e_range', 'max_rmsd', 'mean_rmsd'],
).round(3)

pivot.columns = [f'{v}_{m}' for v, m in pivot.columns]
pivot['e_range_ratio']  = (pivot['e_range_gypsum_style']  / pivot['e_range_rdkit_multi' ]).round(2)
pivot['max_rmsd_ratio'] = (pivot['max_rmsd_gypsum_style'] / pivot['max_rmsd_rdkit_multi']).round(2)

print(pivot.to_string())
print()
print('ratio > 1: gypsum_style is more diverse')
print('ratio < 1: rdkit_multi is more diverse')
